# Playing With PyBullet

This notebook has examples of how to use most of the pybullet classes along with visualization. 

RRT with a single PyBullet Turtle Agent

In [1]:
import sys
sys.path.append('../src')
sys.path.append('../')
sys.path.insert(0, '.')

import importlib  
pybullet_utils = importlib.import_module("pybullet-planning.pybullet_tools.utils")
from Environments import *
from pybullet_env import PyBulletTurtle, PyBulletEnv, CircleObsPybullet
from rrt import RRT
import math

import matplotlib.pyplot as plt


pybullet build time: Nov 28 2023 23:51:11


In [ ]:
# Create a start, goal, obstacles 
goal = (16.5, 5.0)
start = ((7.0, 1.0, 0.1), (0, 0, 0, 1.0))
goal_radius = 1.0
obstacles = [CircleObsPybullet(5, 5, 1),
            CircleObsPybullet(9, 8, 1.5),
            CircleObsPybullet(10, 2.5, 1.5)
            ] 

# Create an env with the obstacles
# Set use_gui to 'False' to run headless, but I've not 
# noticed any performance increase. 
# In the gui, hold Ctrl+left mouse to orbit, Ctrl+center mouse to pan
# Speed is the render rate in Hz. 
# Higher -> more accurate but slower, Lower -> faster but less accurate. 
# 30 is about the floor for reliability 
env = PyBulletEnv(20, 20, obstacles, use_gui=True, speed=30)
# Create an agent and add it to the environment 
# '10' here is the speed. The faster the agent is allowed to move, 
# the quicker planning will be, but again at the expense of accuracy
# 10 is about the ceiling for reasonable accuracy
agent = PyBulletTurtle(10, agent_id=1)
env.add_agent(agent, goal=(goal, goal_radius))

# Run the RRT planner
print("Creating RRT")
rrt  = RRT( start=start, goal=goal, goal_radius=goal_radius, 
           env = env, agent=agent,
           max_iter = 5000, planning_time=math.inf,         
           isvalid_function=PyBulletTurtle.is_new_node_valid, cost_function=PyBulletTurtle.get_cost,
           random_point_function=PyBulletTurtle.get_random_point, 
           reached_goal_function = PyBulletTurtle.agent_reached_goal,
           udf_seed = 77,
           print_flag=False
           )
print("Planning RRT")
rrt.plan_path()

# get the path node ids, the states at each node in a completed path, 
# the control inputs used, and the timesteps of each node in a completed
# path
path_ids, path_states, controls, timesteps = rrt.get_path()
# include this line to gracefully disconnect from GUI
pybullet_utils.disconnect()

Creating RRT
Planning RRT
Total Planning Time for agent 1:  6.390374422073364


In [4]:
# Create an env for visualization
# The speed here again is the refresh rate in Hz for the sim. 
# Pick a speed that allows you to view the env comfortably
env = PyBulletEnv(40, 40, obstacles, speed=2320.)
# add the agent back
env.add_agent(agent, goal=(goal, goal_radius))
# replay the agent's path interactively (see env.replay_path for controls, ENTER key to close)
env.replay_path([rrt.get_high_resolution_path()])
# replay the agent's path semi-interactively, letting PyBullet recreate the dynamics between steps
# (see env.replay_path_dynamics for controls, ENTER key to close)
# env.replay_path_dynamics([path_states], [controls], [timesteps], check_cols=True)

PRRT with up to 6 Turtles

In [1]:
import sys
sys.path.insert(0, '.')
sys.path.insert(0, '../')
sys.path.insert(0, '../src')

import importlib  
pybullet_utils = importlib.import_module("pybullet-planning.pybullet_tools.utils")

import math
from pybullet_env import *
from prrt import PRRT

pybullet build time: Nov 28 2023 23:51:11


In [ ]:
# Create some obstacles. For PRRT with PyBullet
# a special time-dependent obs is not needed, 
# unlike the 'mathematical' sims 
obstacles = [CircleObsPybullet(5, 5, 1),
            CircleObsPybullet(9, 8, 1.5),
            CircleObsPybullet(10, 2.5, 1.5)
            ] 

# create an environment with the given obstacles
env = PyBulletEnv(20, 20, obstacles, use_gui=False, speed=20)
# get a list of agents
num_agents = 3 # increase for more agents, up to 6 without 
               # adding more starts/goals
agents = [PyBulletTurtle(10, agent_id=i) for i in range(num_agents)]
# NOTE: For PRRT, the agents must NOT be added to the env, unlike CRRT

goal = (16.5, 5.0, math.pi, 0.0)
start = ((7.0, 1.0, 0.1), (0, 0, 0, 1.0))

start2 = ((2.5, 1.0, 0.1), (0, 0, 0, 1.0))
goal2 = (12.5, 12.5, math.pi/2, 0.0)

start3 = ((15.0, 2.5, 0.1), (0, 0, 0, 1.0))
goal3 = (2.5, 15.0, math.pi/2, 0.0)

start4 = ((15.0, 10.0, 0.1), (0, 0, 0, 1.0))
goal4 = (1.0, 5.0, math.pi/2, 0.0)

start5 = ((13.5, 7.5, 0.1), (0, 0, 0, 1.0))
goal5 = (5.0, 1.5, math.pi/2, 0.0)

start6 = ((10.0, 16.0, 0.1), (0, 0, 0, 1.0))
goal6 = (13.5, 2.0, math.pi/2, 0.0)

# curate lists for call to PRRT. The indices in each 
# list must match (i.e. all the values for the first agent 
# must be at the beginning of the list, those for the second
# next, etc.)
starts = [start, start2, start3, start4, start5, start6]
goals = [goal, goal2, goal3, goal4, goal5, goal6]
goal_radii = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
isvalid_fucs = [PyBulletTurtle.is_new_node_valid for _ in range(len(agents))]
cost_funcs = [PyBulletTurtle.get_cost for _ in range(len(agents))]
random_pt_funcs = [PyBulletTurtle.get_random_point for _ in range(len(agents))]
reached_goal_funcs = [PyBulletTurtle.agent_reached_goal for _ in range(len(agents))]


(paths, states, controls, timesteps, rrts) = PRRT.plan_multi(agents=agents, 
                                        starts=starts,
                                        goals=goals,
                                        goal_radii=goal_radii,
                                        env=env,
                                        max_iter = math.inf, planning_time=math.inf,         
                                        isvalid_function=isvalid_fucs, 
                                        cost_function=cost_funcs,
                                        random_point_function=random_pt_funcs, 
                                        reached_goal_function = reached_goal_funcs,
                                        udf_seed = 21,
                                        # Tell PRRT to use the PyBullet version of the moving obstacle 
                                        obs_type=AgentObsPybullet
                                        )
pybullet_utils.disconnect()

New Agent
PRRT Goal Reached!
Total Planning Time for agent 0:  4.112607717514038
New Agent
PRRT Goal Reached!
Total Planning Time for agent 1:  15.277251243591309
New Agent
PRRT Goal Reached!
Total Planning Time for agent 2:  34.74464797973633


In [5]:
env = PyBulletEnv(20, 20, obstacles, speed=580.)
# re-add all agents, goals
for i in range(len(agents)):
    env.add_goal(goals[i], goal_radii[i], agent_id=i)
for agent in agents:
    env.add_agent(agent) 
# Both of these sim methods work for multi agent sims as well
env.replay_path([rrt.get_high_resolution_path() for rrt in rrts])
# env.replay_path_dynamics(states, controls, timesteps)

: 

CRRT with up to 6 Turtles

In [1]:
import sys
sys.path.append('../src')
sys.path.append('../')

import importlib  
pybullet_utils = importlib.import_module("pybullet-planning.pybullet_tools.utils")

from pybullet_env import *
from cRRT import CRRT
import math

pybullet build time: Nov 28 2023 23:51:11


In [2]:
obstacles = [CircleObsPybullet(5, 5, 1),
            CircleObsPybullet(9, 8, 1.5),
            CircleObsPybullet(10, 2.5, 1.5)
            ] 

# create an environment with the given obstacles
env = PyBulletEnv(20, 20, obstacles, use_gui=True, speed=20)
# get a list of agents
num_agents = 3 # increase for more agents, up to 6 without 
               # adding more starts/goals
agents = [PyBulletTurtle(10, agent_id=i) for i in range(num_agents)]

goal = (16.5, 5.0, math.pi, 0.0)
start = ((7.0, 1.0, 0.1), (0, 0, 0, 1.0))

start2 = ((2.5, 1.0, 0.1), (0, 0, 0, 1.0))
goal2 = (12.5, 12.5, math.pi/2, 0.0)

start3 = ((15.0, 2.5, 0.1), (0, 0, 0, 1.0))
goal3 = (2.5, 15.0, math.pi/2, 0.0)

start4 = ((15.0, 10.0, 0.1), (0, 0, 0, 1.0))
goal4 = (1.0, 5.0, math.pi/2, 0.0)

start5 = ((13.5, 7.5, 0.1), (0, 0, 0, 1.0))
goal5 = (5.0, 1.5, math.pi/2, 0.0)

start6 = ((10.0, 16.0, 0.1), (0, 0, 0, 1.0))
goal6 = (13.5, 2.0, math.pi/2, 0.0)

# curate lists for call to CRRT. The indices in each 
# list must match (i.e. all the values for the first agent 
# must be at the beginning of the list, those for the second
# next, etc.)
starts = [start, start2, start3, start4, start5, start6]
goals = [goal, goal2, goal3, goal4, goal5, goal6]
goal_radii = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
isvalid_fucs = [PyBulletTurtle.is_new_node_valid for _ in range(len(agents))]
cost_funcs = [PyBulletTurtle.get_cost for _ in range(len(agents))]
random_pt_funcs = [PyBulletTurtle.get_random_point for _ in range(len(agents))]
reached_goal_funcs = [PyBulletTurtle.agent_reached_goal for _ in range(len(agents))]
# For CRRT, all PB agents must be added to the env before planning. 
for i in range(len(agents)):
    env.add_agent(agents[i], goal=(goals[i], goal_radii[i]))

crrt = CRRT(agents=agents, 
                starts=starts,
                goals=goals,
                goal_radii=goal_radii,
                env=env,
                max_iter = math.inf, planning_time=math.inf,         
                isvalid_function=isvalid_fucs, 
                cost_function=cost_funcs,
                random_point_function=random_pt_funcs, 
                reached_goal_function = reached_goal_funcs,
                udf_seed = 77,
                # unlike the mathematical sims, collision checking 
                # in CRRT with PB we get for free. Return false for all
                collision_check_func= lambda planner, paths : False 
                )
# Plan paths
planning_time = crrt.plan_path()

# Get path info
path_rrt_nodes, states, controls, timesteps, costs = crrt.get_path()
pybullet_utils.disconnect()

Total Planning Time:  183.0267457962036


In [4]:
env = PyBulletEnv(20, 20, obstacles, speed=580.)
for i in range(len(agents)):
    env.add_goal(goals[i], goal_radii[i], agent_id=i)
for agent in agents:
    env.add_agent(agent) 
# Once again, both visualization types work here 
env.replay_path(crrt.get_high_resolution_paths())
# env.replay_path_dynamics(states, controls, [timesteps for _ in states])

KCBS with PyBulletTurtles

In [1]:
import sys
sys.path.append('../src')
sys.path.append('../')
sys.path.append('./')

import importlib  
pybullet_utils = importlib.import_module("pybullet-planning.pybullet_tools.utils")

from pybullet_env import *

from constrainedX import *
from kcbs import KCBS
from Environments import *

import math


pybullet build time: Nov 28 2023 23:51:11


In [2]:
obstacles = [CircleObsPybullet(5, 5, 1),
            CircleObsPybullet(9, 8, 1.5),
            CircleObsPybullet(10, 2.5, 1.5)
            ] 

env = PyBulletEnv(20, 20, obstacles, use_gui=True, speed=30)
agents = {}
num_agents = 3
for agent_id in range(num_agents):
    agents[agent_id] = PyBulletTurtle(10, agent_id=agent_id)

goal = (16.5, 5.0, math.pi, 0.0)
start = ((7.0, 1.0, 0.1), (0, 0, 0, 1.0))

start2 = ((2.5, 1.0, 0.1), (0, 0, 0, 1.0))
goal2 = (12.5, 12.5, math.pi/2, 0.0)

start3 = ((15.0, 2.5, 0.1), (0, 0, 0, 1.0))
goal3 = (2.5, 15.0, math.pi/2, 0.0)

start4 = ((15.0, 10.0, 0.1), (0, 0, 0, 1.0))
goal4 = (1.0, 5.0, math.pi/2, 0.0)

start5 = ((13.5, 7.5, 0.1), (0, 0, 0, 1.0))
goal5 = (5.0, 1.5, math.pi/2, 0.0)

start6 = ((10.0, 16.0, 0.1), (0, 0, 0, 1.0))
goal6 = (13.5, 2.0, math.pi/2, 0.0)

starts = [start, start2, start3, start4, start5, start6]
goals = [goal, goal2, goal3, goal4, goal5, goal6]
goal_radii = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
for i in range(len(agents)):
    env.add_agent(agents[i], start=[-10-i, -10-i, 0.2]) # i.e. out of the way
    env.add_goal(goals[i], goal_radii[i], i)

# for KCBS each planner is defined explicitly. Each planner must be 
# beholden to dynamics constraints 
def get_rrt_planner(start,goal,gr,agent):
    rrt_planner = ConstrainedRRTPB( 
            start=start, goal=goal,
            env=env,
            goal_radius=gr, 
            agent=agent,
            sampling_time_step=1.5,
            minimum_time_step=0.1,
            max_iter = math.inf,
            planning_time=math.inf,     
            # Use functions to match the agent    
            isvalid_function=PyBulletTurtle.is_new_node_valid, cost_function=PyBulletTurtle.get_cost,
            random_point_function=PyBulletTurtle.get_random_point, 
            reached_goal_function = PyBulletTurtle.agent_reached_goal,
            print_flag=False,
            udf_seed = 7,
            is_collision_func=env.is_collision
           )
    return rrt_planner

planners = {}
for i in range(len(agents)):
    planners[agents[i].id] = get_rrt_planner(starts[i],goals[i],goal_radii[i],agents[i])

kcbs_planner = KCBS(
                env = env,
                agents = agents,
                low_level_planners = planners,
                max_trials = math.inf,
                planning_time = math.inf,
                is_collision_function=env.is_collision
                )

path_found, paths, path_cost = kcbs_planner.plan_multi_agent_paths()         
pybullet_utils.disconnect()

Total Planning Time for agent 0:  10.877902507781982
Total Planning Time for agent 1:  10.830116033554077
Total Planning Time for agent 2:  15.61184024810791
Current iteration:  0
Total Planning Time for agent 1:  14.198834896087646
Total Planning Time for agent 2:  12.403798341751099
Current iteration:  1


In [4]:
env = PyBulletEnv(20, 20, obstacles, speed=580.)
for i in range(len(agents)):
    env.add_goal(goals[i], goal_radii[i], i)
for agent in agents:
    env.add_agent(agents[agent]) 
# for now, only the full-interactive visualizer is available for KCBS-planned paths
env.replay_path([v for v in paths.values()])

PyBullet Turtle RRT with Edge Bundles

In [ ]:
import sys
sys.path.append('../src')
sys.path.append('../')

import importlib  
pybullet_utils = importlib.import_module("pybullet-planning.pybullet_tools.utils")
from Environments import *
from pybullet_env import PyBulletTurtle, PyBulletEnv, CircleObsPybullet
from edge_bundle_rrt import EdgeBundleRRT

import numpy as np
import math

# load an edge bundle 
from edge_bundle import EdgeBundle
# Must pick an edge bundle that matches the parameters of the agent you plan to use
edge_bundle_file_location = '../edge_bundles/eb_pb_turtle_speed_10_edges-10000.npz' 
data = np.load(edge_bundle_file_location, allow_pickle=True)
eb_pbt = EdgeBundle(data, fix_num_edges=1000)


pybullet build time: Nov 28 2023 23:51:11


In [2]:
goal = (16.5, 5.0, math.pi, 0.0)
start = ((7.0, 1.0, 0.1), (0, 0, 0, 1.0))
goal_radius = 1.0
obstacles = [CircleObsPybullet(5, 5, 1),
            CircleObsPybullet(9, 8, 1.5),
            CircleObsPybullet(10, 2.5, 1.5)
            ] 
                    
env = PyBulletEnv(20, 20, obstacles, use_gui=True, speed=30)
agent = PyBulletTurtle(10, agent_id=1)
env.add_agent(agent, goal=(goal, goal_radius))

print("Creating RRT")
rrt  = EdgeBundleRRT( start=start, goal=goal, goal_radius=goal_radius, 
           env = env, agent=agent, edge_bundle=eb_pbt,
           max_iter = 5000, planning_time=math.inf,         
           isvalid_function=PyBulletTurtle.is_new_node_valid, cost_function=PyBulletTurtle.get_cost,
           random_point_function=PyBulletTurtle.get_random_point, 
           reached_goal_function = PyBulletTurtle.agent_reached_goal,
           translate_function=PyBulletTurtle.point_translate_function,
           udf_seed = 77,
           print_flag=False
           )
print("Planning RRT")
rrt.plan_path()
path_ids, path_states, controls, timesteps = rrt.get_path()
pybullet_utils.disconnect()

Creating RRT
Planning RRT
Total Planning Time for agent 1:  0.9573500156402588


In [5]:
env = PyBulletEnv(40, 40, obstacles, speed=2320.)
env.add_agent(agent, goal=(goal, goal_radius))
# env.replay_path([rrt.get_high_resolution_path()])
env.replay_path_dynamics([path_states], [controls], [timesteps], check_cols=True)

PB EB PRRT

In [1]:
import sys
sys.path.append('../src')
sys.path.append('../')

import importlib  
pybullet_utils = importlib.import_module("pybullet-planning.pybullet_tools.utils")
from Environments import *
from pybullet_env import *
from prrt_eb import EdgeBundlePRRT

import numpy as np
import math
from Agents import UniCycle

from edge_bundle import EdgeBundle
edge_bundle_file_location = '../edge_bundles/eb_pb_turtle_speed_10_edges-10000.npz' 
data = np.load(edge_bundle_file_location, allow_pickle=True)
eb_pbt = EdgeBundle(data, fix_num_edges=1000)


pybullet build time: Nov 28 2023 23:51:11


In [2]:
obstacles = [CircleObsPybullet(5, 5, 1),
            CircleObsPybullet(9, 8, 1.5),
            CircleObsPybullet(10, 2.5, 1.5)
            ] 

env = PyBulletEnv(20, 20, obstacles, use_gui=True, speed=20)
agents = [PyBulletTurtle(10, agent_id=i) for i in range(6)]

goal = (16.5, 5.0, math.pi, 0.0)
start = ((7.0, 1.0, 0.1), (0, 0, 0, 1.0))

start2 = ((2.5, 1.0, 0.1), (0, 0, 0, 1.0))
goal2 = (12.5, 12.5, math.pi/2, 0.0)

start3 = ((15.0, 2.5, 0.1), (0, 0, 0, 1.0))
goal3 = (2.5, 15.0, math.pi/2, 0.0)

start4 = ((15.0, 10.0, 0.1), (0, 0, 0, 1.0))
goal4 = (1.0, 5.0, math.pi/2, 0.0)

start5 = ((13.5, 7.5, 0.1), (0, 0, 0, 1.0))
goal5 = (5.0, 1.5, math.pi/2, 0.0)

start6 = ((10.0, 16.0, 0.1), (0, 0, 0, 1.0))
goal6 = (13.5, 2.0, math.pi/2, 0.0)

# curate lists for call to PRRT. The indices in each 
# list must match (i.e. all the values for the first agent 
# must be at the beginning of the list, those for the second
# next, etc.)
starts = [start, start2, start3, start4, start5, start6]
goals = [goal, goal2, goal3, goal4, goal5, goal6]
goal_radii = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
isvalid_fucs = [PyBulletTurtle.is_new_node_valid for _ in range(len(agents))]
cost_funcs = [PyBulletTurtle.get_cost for _ in range(len(agents))]
random_pt_funcs = [PyBulletTurtle.get_random_point for _ in range(len(agents))]
reached_goal_funcs = [PyBulletTurtle.agent_reached_goal for _ in range(len(agents))]
# additional lists required for 
point_translate_functions = [PyBulletTurtle.point_translate_function for _ in range(len(agents))]
edge_bundles = [eb_pbt for _ in range(len(agents))]

(paths, states, rrts, controls, timesteps) = EdgeBundlePRRT.plan_multi(agents=agents, 
                                        starts=starts,
                                        goals=goals,
                                        goal_radii=goal_radii,
                                        env=env,
                                        edge_bundle=edge_bundles,
                                        max_iter = 150000, planning_time=6000.0,         
                                        isvalid_function=isvalid_fucs, 
                                        cost_function=cost_funcs,
                                        random_point_function=random_pt_funcs, 
                                        reached_goal_function = reached_goal_funcs,
                                        translate_function=point_translate_functions,
                                        udf_seed = 21, obs_type=AgentObsPybullet
                                        )
pybullet_utils.disconnect()

New Agent
Total Planning Time for agent 0:  1.6418287754058838
New Agent
Total Planning Time for agent 1:  3.9235408306121826
New Agent
Total Planning Time for agent 2:  32.679410219192505
New Agent
Total Planning Time for agent 3:  6.077602386474609
New Agent
Total Planning Time for agent 4:  9.88508915901184
New Agent
Total Planning Time for agent 5:  15.887545108795166


In [3]:
env = PyBulletEnv(20, 20, obstacles, speed=580.)
for i in range(len(agents)):
    env.add_goal(goals[i], goal_radii[i], agent_id=i)
for agent in agents:
    env.add_agent(agent) 
env.replay_path([rrt.get_high_resolution_path() for rrt in rrts])

Edge Bundle KCBS with PyBulletTurtles

In [1]:
import sys
sys.path.append('../src')
sys.path.append('../')
sys.path.append('./')

import importlib  
pybullet_utils = importlib.import_module("pybullet-planning.pybullet_tools.utils")
from constrainedX import ConstrainedEdgeBundleRRTPB
from kcbs import KCBS

from pybullet_env import *
from Environments import *

import math

# load an edge bundle 
from edge_bundle import EdgeBundle
# Must pick an edge bundle that matches the parameters of the agent you plan to use
edge_bundle_file_location = '../edge_bundles/eb_pb_turtle_speed_10_edges-10000.npz' 
data = np.load(edge_bundle_file_location, allow_pickle=True)
eb_pbt = EdgeBundle(data, fix_num_edges=1000)


pybullet build time: Nov 28 2023 23:51:11


In [2]:
obstacles = [CircleObsPybullet(5, 5, 1),
            CircleObsPybullet(9, 8, 1.5),
            CircleObsPybullet(10, 2.5, 1.5)
            ] 

env = PyBulletEnv(20, 20, obstacles, use_gui=True, speed=30)
agents = {}
num_agents = 3
for agent_id in range(num_agents):
    agents[agent_id] = PyBulletTurtle(10, agent_id=agent_id)

goal = (16.5, 5.0, math.pi, 0.0)
start = ((7.0, 1.0, 0.1), (0, 0, 0, 1.0))

start2 = ((2.5, 1.0, 0.1), (0, 0, 0, 1.0))
goal2 = (12.5, 12.5, math.pi/2, 0.0)

start3 = ((15.0, 2.5, 0.1), (0, 0, 0, 1.0))
goal3 = (2.5, 15.0, math.pi/2, 0.0)

start4 = ((15.0, 10.0, 0.1), (0, 0, 0, 1.0))
goal4 = (1.0, 5.0, math.pi/2, 0.0)

start5 = ((13.5, 7.5, 0.1), (0, 0, 0, 1.0))
goal5 = (5.0, 1.5, math.pi/2, 0.0)

start6 = ((10.0, 16.0, 0.1), (0, 0, 0, 1.0))
goal6 = (13.5, 2.0, math.pi/2, 0.0)

starts = [start, start2, start3, start4, start5, start6]
goals = [goal, goal2, goal3, goal4, goal5, goal6]
goal_radii = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
for i in range(len(agents)):
    env.add_agent(agents[i], start=[-10-i, -10-i, 0.2]) # i.e. out of the way
    env.add_goal(goals[i], goal_radii[i], i)

# for KCBS each planner is defined explicitly. Each planner must be 
# beholden to dynamics constraints 
def get_rrt_planner(start,goal,gr,agent):
    rrt_planner = ConstrainedEdgeBundleRRTPB( 
            start=start, goal=goal,
            env=env,
            goal_radius=gr, 
            agent=agent,
            edge_bundle = eb_pbt, 
            sampling_time_step=1.5,
            minimum_time_step=0.1,
            max_iter = math.inf,
            num_random_edges= 10,
            num_skip_edges= 100,
            planning_time=math.inf,     
            # Use functions to match the agent    
            isvalid_function=PyBulletTurtle.is_new_node_valid, cost_function=PyBulletTurtle.get_cost,
            random_point_function=PyBulletTurtle.get_random_point, 
            reached_goal_function = PyBulletTurtle.agent_reached_goal,
            translate_function=PyBulletTurtle.point_translate_function,
            print_flag=False,
            udf_seed = 7,
            is_collision_func=env.is_collision
           )
    return rrt_planner

planners = {}
for i in range(len(agents)):
    planners[agents[i].id] = get_rrt_planner(starts[i],goals[i],goal_radii[i],agents[i])

kcbs_planner = KCBS(
                env = env,
                agents = agents,
                low_level_planners = planners,
                max_trials = math.inf,
                planning_time = math.inf,
                is_collision_function=env.is_collision
                )

path_found, paths, path_cost = kcbs_planner.plan_multi_agent_paths()         
pybullet_utils.disconnect()

Total Planning Time for agent 0:  1.8465652465820312
Total Planning Time for agent 1:  1.9145240783691406
Total Planning Time for agent 2:  2.682309150695801
Current iteration:  0


In [3]:
env = PyBulletEnv(20, 20, obstacles, speed=580.)
for i in range(len(agents)):
    env.add_goal(goals[i], goal_radii[i], i)
for agent in agents:
    env.add_agent(agents[agent]) 
# for now, only the full-interactive visualizer is available for KCBS-planned paths
env.replay_path([v for v in paths.values()])